# Convert a pandas dataframe to geojson for web-mapping

Author: Geoff Boeing

Original: [pandas-to-geojson](https://github.com/gboeing/urban-data-science/blob/dc86c9c89b73f87f97301883d7456f1f814589f5/17-Leaflet-Web-Mapping/pandas-to-geojson.ipynb)

In [ ]:
import json
import requests
import pandas as pd

Download a small sample of NYC 311 data from the public Socrata API. The query below limits the response to mappable rows and only requests the columns this notebook needs.

In [ ]:
endpoint_url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
query_params = {
    "$limit": 20,
    "$select": ",".join(
        [
            "complaint_type",
            "descriptor",
            "status",
            "incident_address",
            "street_name",
            "intersection_street_1",
            "intersection_street_2",
            "city",
            "borough",
            "latitude",
            "longitude",
        ]
    ),
    "$where": "latitude is not null and longitude is not null and status is not null",
}

In [ ]:
response = requests.get(endpoint_url, params=query_params, timeout=30)
response.raise_for_status()
data = response.json()

In [ ]:
df = pd.DataFrame(data)
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

intersection_address = (
    df["intersection_street_1"]
    .fillna("")
    .str.strip()
    .str.cat(df["intersection_street_2"].fillna("").str.strip(), sep=" & ")
    .str.strip(" &")
    .replace("", pd.NA)
)

df["display_address"] = (
    df["incident_address"]
    .fillna(df["street_name"])
    .fillna(intersection_address)
    .fillna("Unknown location")
    .str.title()
)

In [ ]:
cols = [
    "complaint_type",
    "descriptor",
    "latitude",
    "longitude",
    "display_address",
    "status",
    "borough",
    "city",
]
df_subset = df[cols]
df_geo = df_subset.dropna(subset=["latitude", "longitude"], axis=0, inplace=False)

In [ ]:
def df_to_geojson(df, properties, lat="latitude", lon="longitude"):
    geojson = {"type": "FeatureCollection", "features": []}

    for _, row in df.iterrows():
        feature = {
            "type": "Feature",
            "properties": {},
            "geometry": {"type": "Point", "coordinates": []},
        }
        feature["geometry"]["coordinates"] = [row[lon], row[lat]]
        for prop in properties:
            feature["properties"][prop] = row[prop]
        geojson["features"].append(feature)

    return geojson

In [ ]:
cols = ["display_address", "city", "borough", "complaint_type", "descriptor", "status"]
geojson = df_to_geojson(df_geo, cols)
geojson["features"][0]

In nteract, this geojson can be rendered directly with the built-in leaflet renderer.

In [ ]:
import IPython

IPython.display.display({"application/geo+json": geojson}, raw=True)